1.1 중복 및 유사 데이터 제거 코드
이 코드는 link 기준의 물리적 중복을 먼저 지우고, 자카드 유사도를 이용해 의미적 중복을 2차로 지웁니다.

In [1]:
import pandas as pd
from tqdm import tqdm

# [1] 데이터 로드 및 초기 상태 기록
df = pd.read_csv('dataset.csv')
original_cnt = len(df)

# [2] 물리적 중복 제거 (링크 기준)
df_unique_link = df.drop_duplicates(subset=['link'], keep='first').copy()
link_dedup_cnt = len(df_unique_link)

# [3] 유사도 기반 제거 준비 (본문 길이순 정렬 -> 긴 것 보존)
df_unique_link['text_len'] = df_unique_link['content'].str.len()
df_sorted = df_unique_link.sort_values(by='text_len', ascending=False).reset_index(drop=True)

# 자카드 유사도 함수 정의
def get_jaccard_sim(str1, str2):
    a = set(str(str1).split())
    b = set(str(str2).split())
    union = len(a.union(b))
    return len(a.intersection(b)) / union if union > 0 else 0

# [4] 유사도 비교 및 제거 (80% 기준)
to_drop = set()
threshold = 0.8

for i in tqdm(range(len(df_sorted)), desc="유사도 검사 중"):
    if i in to_drop: continue
    for j in range(i + 1, len(df_sorted)):
        if j in to_drop: continue
        # 제목 + 본문 유사도 측정
        sim = get_jaccard_sim(df_sorted.iloc[i]['title'] + " " + df_sorted.iloc[i]['content'], 
                              df_sorted.iloc[j]['title'] + " " + df_sorted.iloc[j]['content'])
        if sim >= threshold:
            to_drop.add(j)

# 최종 Step 1 결과 데이터프레임
df_step1 = df_sorted.drop(list(to_drop)).drop(columns=['text_len']).reset_index(drop=True)
step1_final_cnt = len(df_step1)

유사도 검사 중: 100%|███████████████████████████████████████████████████████████| 1225/1225 [03:11<00:00,  6.39it/s]



자카드 유사도(Jaccard Similarity)는 두 집합 사이의 유사도를 측정하는 가장 대표적인 방법 중 하나입니다. 데이터 사이언스에서 특히 텍스트 중복 제거나 추천 시스템에서 자주 활용됩니다.

1. 정의 및 공식
자카드 유사도는 두 집합의 교집합 크기를 합집합의 크기로 나눈 값입니다.

J(A,B)= 
∣A∪B∣
∣A∩B∣
​
 = 
∣A∣+∣B∣−∣A∩B∣
∣A∩B∣
​
 
값의 범위: 0에서 1 사이

1: 두 집합이 완전히 동일함

0: 두 집합 사이에 공통 원소가 하나도 없음

2. 작동 원리 (예시)
두 개의 뉴스 문장이 있다고 가정해 보겠습니다.

문장 A: "현대자동차 수입 단가 상승" → 집합 A: {현대자동차, 수입, 단가, 상승}

문장 B: "현대자동차 수입 비용 증가" → 집합 B: {현대자동차, 수입, 비용, 증가}

교집합 (A∩B): {현대자동차, 수입} (크기: 2)

합집합 (A∪B): {현대자동차, 수입, 단가, 상승, 비용, 증가} (크기: 6)

자카드 유사도: 2/6=0.33 (약 33%)

3. 주요 특징
단어의 순서를 무시: 집합(Set)을 기반으로 하기 때문에 문장 내 단어 순서가 달라도 구성 단어가 같다면 유사도가 1로 나옵니다.

단어의 빈도수를 무시: 단어가 문장에 여러 번 등장해도 집합에서는 하나로 취급합니다. (빈도까지 고려하려면 '코사인 유사도' 등을 사용합니다.)

직관적임: 두 문서가 얼마나 많은 단어를 공유하는지 비율로 나타내므로 이해하기 매우 쉽습니다.



1.2 Step 1 중간 결과 확인

In [2]:
print("📊 [Step 1] 중복/유사 데이터 정제 리포트")
print("-" * 40)
print(f"• 원본 데이터 개수: {original_cnt}건")
print(f"• 1차 제거 (링크 중복): {original_cnt - link_dedup_cnt}건 제거")
print(f"• 2차 제거 (유사도 80%): {link_dedup_cnt - step1_final_cnt}건 제거")
print(f"• 현재 남은 데이터: {step1_final_cnt}건")
print("-" * 40)

# 제거된 데이터의 예시를 보고 싶다면?
if len(to_drop) > 0:
    print("\n🧐 유사 기사로 판명되어 제거된 샘플 (일부):")
    dropped_idx = list(to_drop)[0]
    print(f"이미 남겨진 기사: {df_sorted.iloc[0]['title'][:50]}...")
    print(f"제거된 유사 기사: {df_sorted.iloc[dropped_idx]['title'][:50]}...")

📊 [Step 1] 중복/유사 데이터 정제 리포트
----------------------------------------
• 원본 데이터 개수: 1275건
• 1차 제거 (링크 중복): 50건 제거
• 2차 제거 (유사도 80%): 45건 제거
• 현재 남은 데이터: 1180건
----------------------------------------

🧐 유사 기사로 판명되어 제거된 샘플 (일부):
이미 남겨진 기사: 로봇은 왜 인간의 모습을 하고 있는가?...
제거된 유사 기사: 코스피 2.97% 하락한 5277.30 마감...반도체·자동차 약세...


2.1 노이즈 및 이모지 정밀 제거

In [3]:
import re

def clean_news_step2(text):
    if not isinstance(text, str): return ""
    
    # 1. 날짜 및 시간 패턴 제거 (예: 2026-03-31 09:36)
    text = re.sub(r'\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}', '', text)
    
    # 2. SNS 공유 문구 제거 (재훈님 요청사항 반영)
    share_pattern = r'카카오톡\s?페이스북\s?엑스\s?URL공유|URL\s?공유'
    text = re.sub(share_pattern, '', text)
    
    # 3. 저작권 및 상투적인 문구 제거
    copyright_patterns = [r'무단전재 및 재배포 금지', r'Copyright.*?All rights reserved', r'저작권자 ⓒ.*?']
    for p in copyright_patterns:
        text = re.sub(p, '', text, flags=re.IGNORECASE)

    # 4. 이모지 및 불필요한 특수 기호 제거 (★, ◆, ⚡ 등)
    # 한글, 숫자, 영어, 기본 문장부호, 그리고 중요 기호(%, $)만 허용합니다.
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.,!\?\(\)\[\]%$\u20A9]', '', text)

    # 5. 언론사/기자 괄호 정보 제거 (예: [서울=뉴스1])
    text = re.sub(r'\[.*?\]|\(.*?\)', '', text)

    # 6. 연속된 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# [실행] Step 1 결과물에 적용
df_step2 = df_step1.copy()
df_step2['content_cleaned'] = df_step2['content'].apply(clean_news_step2)

# [필터링] 청소 후 내용이 너무 짧아진(20자 미만) 쓰레기 데이터 제거
before_cnt = len(df_step2)
df_step2 = df_step2[df_step2['content_cleaned'].str.len() >= 20].reset_index(drop=True)
after_cnt = len(df_step2)

2.2. 중간결과확인

In [4]:
print("📊 [Step 2] 노이즈/이모지 정제 리포트")
print("-" * 40)
print(f"• 정제 전 데이터: {before_cnt}건")
print(f"• 부실 데이터 제거(20자 미만): {before_cnt - after_cnt}건")
print(f"• 최종 남은 데이터: {after_cnt}건")
print("-" * 40)

# 실제 텍스트 변화 확인 (첫 번째 기사 기준)
print("\n📝 텍스트 정제 샘플 비교:")
print(f"원본 일부: {df_step1['content'].iloc[0][:80]}...")
print(f"정제 후:  {df_step2['content_cleaned'].iloc[0][:80]}...")

# 중요 기호(%) 생존 확인
if '%' in df_step2['content_cleaned'].iloc[0]:
    print("\n✅ 확인: 퍼센트(%) 기호가 안전하게 보존되었습니다!")

📊 [Step 2] 노이즈/이모지 정제 리포트
----------------------------------------
• 정제 전 데이터: 1180건
• 부실 데이터 제거(20자 미만): 1건
• 최종 남은 데이터: 1179건
----------------------------------------

📝 텍스트 정제 샘플 비교:
원본 일부: 분야별 TV 매거진 [리더십 특강] 한양대 공학대학 로봇공학과 한재권 교수 한재권 교수는 한양대학교 로봇공학과 부교수이자, 휴머노이드 전문 스타...
정제 후:  분야별 TV 매거진 한양대 공학대학 로봇공학과 한재권 교수 한재권 교수는 한양대학교 로봇공학과 부교수이자, 휴머노이드 전문 스타트업 AeiROB...


In [5]:
import pandas as pd

# 1. 확인하고 싶은 노이즈 키워드 목록
check_keywords = ['카카오톡', '페이스북', '엑스', 'URL공유', '재배포 금지', '2026-03-31']

print("🔍 [노이즈 제거 확증 리포트]")
print("-" * 50)

for kw in check_keywords:
    # 원본(content)에는 있지만, 정제본(content_cleaned)에는 없는 행 찾기
    removed_mask = df_step2['content'].str.contains(kw) & ~df_step2['content_cleaned'].str.contains(kw)
    removed_count = removed_mask.sum()
    
    print(f"• 키워드 '{kw}': {removed_count}개의 기사에서 제거됨")
    
    # 제거된 실제 사례 하나만 출력해서 확인하기
    if removed_count > 0:
        sample_idx = df_step2[removed_mask].index[0]
        # 키워드 주변 30자씩만 잘라서 비교 (눈에 잘 띄게)
        start_pos = df_step2.loc[sample_idx, 'content'].find(kw)
        context_before = df_step2.loc[sample_idx, 'content'][max(0, start_pos-20):start_pos+len(kw)+20]
        
        print(f"  └─ [원본 샘플]: ...{context_before}...")
        print(f"  └─ [정제 결과]: (해당 문구 삭제됨)")
    print("-" * 50)

# 2. 전체적인 텍스트 길이 변화 확인
avg_before = df_step2['content'].str.len().mean()
avg_after = df_step2['content_cleaned'].str.len().mean()

print(f"\n📊 텍스트 밀도 변화")
print(f"• 평균 글자 수: {avg_before:.1f}자 -> {avg_after:.1f}자 (약 {avg_before - avg_after:.1f}자 노이즈 제거됨)")

🔍 [노이즈 제거 확증 리포트]
--------------------------------------------------
• 키워드 '카카오톡': 6개의 기사에서 제거됨
  └─ [원본 샘플]: ...입력 2026-03-31 08:23 카카오톡 페이스북 엑스 URL공유 가장작게 ...
  └─ [정제 결과]: (해당 문구 삭제됨)
--------------------------------------------------
• 키워드 '페이스북': 6개의 기사에서 제거됨
  └─ [원본 샘플]: ...26-03-31 08:23 카카오톡 페이스북 엑스 URL공유 가장작게 작게 기본...
  └─ [정제 결과]: (해당 문구 삭제됨)
--------------------------------------------------
• 키워드 '엑스': 12개의 기사에서 제거됨
  └─ [원본 샘플]: ...-31 08:23 카카오톡 페이스북 엑스 URL공유 가장작게 작게 기본 크게...
  └─ [정제 결과]: (해당 문구 삭제됨)
--------------------------------------------------
• 키워드 'URL공유': 6개의 기사에서 제거됨
  └─ [원본 샘플]: ... 08:23 카카오톡 페이스북 엑스 URL공유 가장작게 작게 기본 크게 가장크게 ...
  └─ [정제 결과]: (해당 문구 삭제됨)
--------------------------------------------------
• 키워드 '재배포 금지': 11개의 기사에서 제거됨
  └─ [원본 샘플]: ...edia.co.kr), 무단전재 및 재배포 금지] (조세금융신문=안종명 기자) 지난...
  └─ [정제 결과]: (해당 문구 삭제됨)
--------------------------------------------------
• 키워드 '2026-03-31': 8개의 기사에서 제거됨
  └─ [원본 샘플]: ...입력 2026-03-31 08:23 카카오톡 페이스북 엑스

3.1 현대자동차 명칭 단일화

In [6]:
import re

# 1. 통일할 대상 패턴 정의 (긴 단어부터 배치하여 부분 일치 오류 방지)
# 현대차그룹, 현대차, Hyundai motor, Hyundai 등을 모두 포함합니다.
hyundai_pattern = r'현대차그룹|현대차|Hyundai [Mm]otor|Hyundai'

def unify_hyundai_name(text):
    if not isinstance(text, str): return ""
    
    # 패턴에 해당되는 모든 단어를 '현대자동차'로 변경
    return re.sub(hyundai_pattern, '현대자동차', text)

# 2. Step 2 결과물에 적용
df_step3 = df_step2.copy()

# 변경 전 빈도 측정을 위해 임시 저장
before_counts = {
    '현대차': df_step3['content_cleaned'].str.contains('현대차').sum(),
    'Hyundai': df_step3['content_cleaned'].str.contains('Hyundai', case=False).sum()
}

# 명칭 통일 적용
df_step3['content_final'] = df_step3['content_cleaned'].apply(unify_hyundai_name)

# 변경 후 빈도 측정
after_count = df_step3['content_final'].str.contains('현대자동차').sum()

3.2 중간 결과 확인 (검증 리포트)

In [7]:
print("📊 [Step 3] 명칭 통일(Entity Unification) 리포트")
print("-" * 50)
print(f"• 변경 전 '현대차' 포함 기사: {before_counts['현대차']}건")
print(f"• 변경 전 'Hyundai' 포함 기사: {before_counts['Hyundai']}건")
print("-" * 50)
print(f"• 변경 후 '현대자동차' 통합 기사: {after_count}건")
print("-" * 50)

# 실제 문장 변화 샘플 확인
sample_mask = df_step2['content_cleaned'].str.contains('Hyundai|현대차', case=False)
if sample_mask.any():
    idx = df_step2[sample_mask].index[0]
    print("\n📝 명칭 통일 샘플 비교:")
    print(f"└─ [전]: {df_step2.loc[idx, 'content_cleaned'][:60]}...")
    print(f"└─ [후]: {df_step3.loc[idx, 'content_final'][:60]}...")

📊 [Step 3] 명칭 통일(Entity Unification) 리포트
--------------------------------------------------
• 변경 전 '현대차' 포함 기사: 511건
• 변경 전 'Hyundai' 포함 기사: 1건
--------------------------------------------------
• 변경 후 '현대자동차' 통합 기사: 1146건
--------------------------------------------------

📝 명칭 통일 샘플 비교:
└─ [전]: 분야별 TV 매거진 한양대 공학대학 로봇공학과 한재권 교수 한재권 교수는 한양대학교 로봇공학과 부교수이자, ...
└─ [후]: 분야별 TV 매거진 한양대 공학대학 로봇공학과 한재권 교수 한재권 교수는 한양대학교 로봇공학과 부교수이자, ...


4.1 스포츠 및 비즈니스 외 뉴스 필터링

In [8]:
# 1. 제거할 키워드 정의 (스포츠, 경기 결과 등 공급망과 관련 없는 단어들)
exclude_keywords = [
    '배구', '농구', '축구', '야구', '골프', '양궁', # 스포츠 종목
    '리그', '경기 결과', '승리', '패배', '득점', '시즌', # 경기 관련
    '피버스', '스카이워커스', '현대건설 배구단', '전북현대', # 스포츠 팀명
    '스폰서십', '네이밍 스폰서' # 단순 후원 관련
]

# 2. 필터링 함수 정의
def is_sports_news(row):
    # 제목이나 본문에 스포츠 관련 키워드가 있는지 확인
    combined_text = f"{row['title']} {row['content_final']}"
    for word in exclude_keywords:
        if word in combined_text:
            return True
    return False

# 3. 필터링 적용
df_step4 = df_step3.copy()
sports_mask = df_step4.apply(is_sports_news, axis=1)

# 스포츠 뉴스로 분류된 데이터와 비즈니스 데이터 분리
df_filtered_business = df_step4[~sports_mask].reset_index(drop=True)
df_removed_sports = df_step4[sports_mask]

# 4. 결과 보고
print("📊 [Step 4] 도메인 외(스포츠 등) 데이터 정제 리포트")
print("-" * 50)
print(f"• 정제 전 데이터: {len(df_step4)}건")
print(f"• 스포츠/비관련 뉴스 제거: {len(df_removed_sports)}건")
print(f"• 최종 공급망 분석 데이터: {len(df_filtered_business)}건")
print("-" * 50)

📊 [Step 4] 도메인 외(스포츠 등) 데이터 정제 리포트
--------------------------------------------------
• 정제 전 데이터: 1179건
• 스포츠/비관련 뉴스 제거: 45건
• 최종 공급망 분석 데이터: 1134건
--------------------------------------------------


4.2 중간 결과 확인

In [9]:
if len(df_removed_sports) > 0:
    print("\n🚫 제거된 스포츠/비관련 뉴스 샘플:")
    for i in range(min(3, len(df_removed_sports))):
        print(f"└─ 제목: {df_removed_sports.iloc[i]['title']}")
else:
    print("\n✅ 현재 데이터셋에는 관련 없는 스포츠 뉴스가 발견되지 않았습니다.")


🚫 제거된 스포츠/비관련 뉴스 샘플:
└─ 제목: [결정의 순간들②] 정몽규 인생 최고의 순간은 '기아차 인수'
└─ 제목: 손흥민, '<b>현대차</b> 글로벌 앰배서더' 됐다…찰칵 세리머니 '아틀라스'도 전...
└─ 제목: [거래소 외국인] 삼성SDI·현대건설 '사자' vs SK하이닉스·삼성전자 '팔...


In [10]:
import os

# 1. 저장할 파일명 정의
final_output_file = "hyundai_supply_chain_news_final.csv"

# 2. CSV 파일로 추출
# index=False: 데이터프레임의 인덱스 번호는 제외하고 저장합니다.
# encoding='utf-8-sig': 엑셀에서 한글이 깨지지 않게 해주는 가장 중요한 설정입니다.
df_filtered_business.to_csv(final_output_file, index=False, encoding='utf-8-sig')

# 3. 저장 결과 및 위치 확인
full_path = os.path.abspath(final_output_file)

print("🎊 [최종 추출 완료] 🎊")
print("-" * 50)
print(f"• 최종 데이터 개수: {len(df_filtered_business)}건")
print(f"• 저장된 파일명: {final_output_file}")
print(f"• 실제 저장 경로: {full_path}")
print("-" * 50)

# (참고) 데이터가 잘 들어갔는지 마지막으로 5개만 확인
# print(df_filtered_business[['title', 'content_final']].head())

🎊 [최종 추출 완료] 🎊
--------------------------------------------------
• 최종 데이터 개수: 1134건
• 저장된 파일명: hyundai_supply_chain_news_final.csv
• 실제 저장 경로: C:\Users\JH\hyundai_supply_chain_news_final.csv
--------------------------------------------------


In [11]:
import pandas as pd

# 1. 이전 단계에서 생성된 최종 파일 로드
file_path = "hyundai_supply_chain_news_final.csv"
df_final = pd.read_csv(file_path)

# 2. 불필요한 컬럼 제거 및 이름 변경
# 'Unnamed: 0', 'content'(원본), 'content_cleaned'(중간정제본) 삭제
df_organized = df_final.drop(columns=['Unnamed: 0', 'content', 'content_cleaned'])

# 3. 'content_final' 컬럼을 'content'로 이름 변경
df_organized = df_organized.rename(columns={'content_final': 'content'})

# 4. 컬럼 순서 맞추기 (이미지의 dataset.csv 형식과 동일하게: title, link, pub_date, content)
# 분석 환경에 따라 컬럼 순서가 중요할 수 있으므로 명시적으로 재배치합니다.
target_order = ['title', 'link', 'pub_date', 'content']
df_organized = df_organized[target_order]

# 5. 최종 결과 저장
output_v2 = "hyundai_supply_chain_news_final_v2.csv"
df_organized.to_csv(output_v2, index=False, encoding='utf-8-sig')

print("✨ [최종 데이터 규격 정리 완료] ✨")
print("-" * 50)
print(f"• 정리된 컬럼: {', '.join(df_organized.columns)}")
print(f"• 저장된 파일명: {output_v2}")
print("-" * 50)

# 제대로 바뀌었는지 상위 3개만 출력해서 확인
print(df_organized.head(3))

✨ [최종 데이터 규격 정리 완료] ✨
--------------------------------------------------
• 정리된 컬럼: title, link, pub_date, content
• 저장된 파일명: hyundai_supply_chain_news_final_v2.csv
--------------------------------------------------
                                           title  \
0                          로봇은 왜 인간의 모습을 하고 있는가?   
1               [결정의 순간들①] 정주영·정세영의 유산 'HDC 정몽규'   
2  [이슈체크] 이란전쟁 한달 만에 시총 840조 증발…'삼전닉스'만 372조원...   

                                                link  \
0          http://www.brainmedia.co.kr/Opinion/25313   
1  https://www.meconomynews.com/news/articleView....   
2  https://www.tfmedia.co.kr/news/article.html?no...   

                          pub_date  \
0  Tue, 31 Mar 2026 08:44:00 +0900   
1  Tue, 31 Mar 2026 09:38:00 +0900   
2  Tue, 31 Mar 2026 08:50:00 +0900   

                                             content  
0  분야별 TV 매거진 한양대 공학대학 로봇공학과 한재권 교수 한재권 교수는 한양대학교...  
1  정몽규 회장은 대한민국 재벌이다. 고 정주영 회장의 조카로 태어나 현대자동차 회장으...  
2  2026.03.31 이란전쟁 여파로 3월 한 달간 국내